# Final Coontributions: Exploring Public Transportation Usage by Neighborhood

**Author:** Abigail Adams 

**Group:** Abigail Adams, Jasmine Law, Caleb Huang and Ronald Tien Giang

**Notebook Purpose:** This notebook explores modes of transportation used by San Francisco residents. The purpose of this notebook is to identify the percentage of people in San Francisco using public transportation. The majority of this notebook will be cleaning the ACS dataset for future demographic projects. 

**Data Sources:** 
1. American Community Survey [(ACS)](https://www.socialexplorer.com/reports/socialexplorer/en/report/c529bd86-ffc7-11ef-8f8b-e71b92843544) 5-year estimate (2019-2023)

**Steps:** 

* I will clean and run summary statistics on the major modes of transportation using the ACS data.

## Part 1: ACS  
To determine the most commonly used modes of transportation, we will use data from the American Community Survey (ACS), specifically the 5-year estimate from 2017-2023. This is the most recent ACS data available. The data is scaled by census tracts. 

**Variables Included:**

1. Means of Transportation to Work for Workers 16 Years and Over
2. Total Population
3. Age (Short version)
4. Race
5. Hispanic or Latino by Race
6. Housing Units by Families and Presence of Own Children
7. Median Household Income (In 2023 Inflation Adjusted Dollars)
8. Median Family Income (In 2023 Inflation Adjusted Dollars)
10. Housing Units by Vehicles Available


In [1]:
# import necessary packages
# to read and wrangle data
import pandas as pd

# to import data from LA Data portal
from sodapy import Socrata

# to create spatial data
import geopandas as gpd

# for basemaps
import contextily as ctx

# For spatial statistics
import esda
from esda.moran import Moran, Moran_Local

import splot
from splot.esda import moran_scatterplot, plot_moran, lisa_cluster,plot_moran_simulation

import libpysal as lps

# Graphics
import matplotlib.pyplot as plt
import plotly.express as px

In [2]:
# import acs data
acs = pd.read_csv("data/up221_wk7_data.csv")

# verify download and check dimensions
acs.shape

(245, 153)

The ACS dataset has 245 rows and 153 columns. Each row is a census tract and each column represents a variable. Our next step is to cleaning the dataset to properly name the columns. 

In [3]:
# check first 5 rows of data 
acs.head(1)

,FIPS,Qualifying Name,State Postal Abbreviation,Summary Level,Geographic Component,US,Census Region,Census Division,State (FIPS Code),County of current residence,...,Occupied Housing Units: 2 Vehicles Available,Occupied Housing Units: 3 Vehicles Available,Occupied Housing Units: 4 Vehicles Available,Occupied Housing Units: 5 or More Vehicles Available,% Occupied Housing Units: No Vehicle Available,% Occupied Housing Units: 1 Vehicle Available,% Occupied Housing Units: 2 Vehicles Available,% Occupied Housing Units: 3 Vehicles Available,% Occupied Housing Units: 4 Vehicles Available,% Occupied Housing Units: 5 or More Vehicles Available
0,Geo_FIPS,Geo_QName,Geo_STUSAB,Geo_SUMLEV,Geo_GEOCOMP,Geo_US,Geo_REGION,Geo_DIVISION,Geo_STATE,Geo_COUNTY,...,SE_A10030_004,SE_A10030_005,SE_A10030_006,SE_A10030_007,PCT_SE_A10030_002,PCT_SE_A10030_003,PCT_SE_A10030_004,PCT_SE_A10030_005,PCT_SE_A10030_006,PCT_SE_A10030_007


**Column Names**

Note that the columns of this dataset are already named according to their variables. The typical variable names are located in row 0. We will ensure to remove Row 0 from our analysis because the values are different and can change the variable's data type. 

This prevents us from having to rename over 100 columns, but may affect the data types of each columns. 

In [4]:
# remove row 0
acs.drop(acs.head(1).index,inplace=True)

# check
acs.head(2)

,FIPS,Qualifying Name,State Postal Abbreviation,Summary Level,Geographic Component,US,Census Region,Census Division,State (FIPS Code),County of current residence,...,Occupied Housing Units: 2 Vehicles Available,Occupied Housing Units: 3 Vehicles Available,Occupied Housing Units: 4 Vehicles Available,Occupied Housing Units: 5 or More Vehicles Available,% Occupied Housing Units: No Vehicle Available,% Occupied Housing Units: 1 Vehicle Available,% Occupied Housing Units: 2 Vehicles Available,% Occupied Housing Units: 3 Vehicles Available,% Occupied Housing Units: 4 Vehicles Available,% Occupied Housing Units: 5 or More Vehicles Available
1,06075010101,"Census Tract 101.01, San Francisco County, Cal...",ca,140,00,NaN,NaN,NaN,06,075,...,102,0,0,0,37.87,53.71,8.42,0,0,0
2,06075010102,"Census Tract 101.02, San Francisco County, Cal...",ca,140,00,NaN,NaN,NaN,06,075,...,148,10,0,0,33.41,49.17,16.32,1.1,0,0


In [5]:
# make object for column names, too long to print
column_names = acs.columns.tolist()

# print
column_names

['FIPS',
 'Qualifying Name',
 'State Postal Abbreviation',
 'Summary Level',
 'Geographic Component',
 'US',
 'Census Region',
 'Census Division',
 'State (FIPS Code)',
 'County of current residence',
 'County Subdivision (FIPS)',
 'Place (FIPS Code)',
 'Census Tract',
 'Block Group',
 'Consolidated City',
 'American Indian Area/Alaska Native Area/Hawaiian Home Land (Census)',
 'American Indian Area/Alaska Native Area/Hawaiian Home Land (FIPS)',
 'American Indian Trust Land/Hawaiian Home Land Indicator',
 'American Indian Tribal Subdivision (FIPS)',
 'Alaska Native Regional Corporation (FIPS)',
 'Metropolitan and Micropolitan Statistical Area',
 'Combined Statistical Area',
 'Metropolitan Statistical Area Metropolitan Division',
 'Metropolitan Area Central City',
 'Metropolitan/Micropolitan Indicator Flag',
 'New England City and Town Area',
 'New England City and Town Combined Statistical Area',
 'New England City and Town Area Division',
 'Urban Area',
 'Current Congressional Distric

**Column Removal**

Now, we will be removing columns that will not be used for this notebook. While there are only 8 variables, the columns also make up each individual answer option. We only need about half of these columns to get information that is useful for this project. 

At a minimum, we will be removing:

* Columns 2-15 because they are geographic components that make up the `FIPS` column
* Empty columns (all NA values)

In [6]:
# print columns with no values
na_columns = acs.columns[acs.isnull().all()].tolist()

na_columns

#check = acs.dropna(axis=1,how="all")

['US',
 'Census Region',
 'Census Division',
 'County Subdivision (FIPS)',
 'Place (FIPS Code)',
 'Block Group',
 'Consolidated City',
 'American Indian Area/Alaska Native Area/Hawaiian Home Land (Census)',
 'American Indian Area/Alaska Native Area/Hawaiian Home Land (FIPS)',
 'American Indian Trust Land/Hawaiian Home Land Indicator',
 'American Indian Tribal Subdivision (FIPS)',
 'Alaska Native Regional Corporation (FIPS)',
 'Metropolitan and Micropolitan Statistical Area',
 'Combined Statistical Area',
 'Metropolitan Statistical Area Metropolitan Division',
 'Metropolitan Area Central City',
 'Metropolitan/Micropolitan Indicator Flag',
 'New England City and Town Area',
 'Urban Area',
 'Current Congressional District ***',
 'State Legislative District Upper',
 'State Legislative District Lower',
 'ZIP Code Tabulation Area (3-digit)',
 '5-digit ZIP Code Tabulation Area',
 'Subminor Civil Division (FIPS)',
 'State-School District (Elementary)',
 'State-School District (Secondary)',
 'S

The object `na_columns` tells us which columns to drop. 

In [7]:
# remove columns with na from object 'na_columns' + unnecessary geo columns
acs_trim = acs.drop(['Qualifying Name',
 'State Postal Abbreviation',
 'Summary Level',
 'Geographic Component',
 'US',
 'Census Region',
 'Census Division',
 'State (FIPS Code)',
 'County of current residence',
 'County Subdivision (FIPS)',
 'Place (FIPS Code)',
 #'Census Tract',
 'Block Group',
 'Consolidated City',
 'American Indian Area/Alaska Native Area/Hawaiian Home Land (Census)',
 'American Indian Area/Alaska Native Area/Hawaiian Home Land (FIPS)',
 'American Indian Trust Land/Hawaiian Home Land Indicator',
 'American Indian Tribal Subdivision (FIPS)',
 'Alaska Native Regional Corporation (FIPS)',
 'Metropolitan and Micropolitan Statistical Area',
 'Combined Statistical Area',
 'Metropolitan Statistical Area Metropolitan Division',
 'Metropolitan Area Central City',
 'Metropolitan/Micropolitan Indicator Flag',
 'New England City and Town Area',
 'New England City and Town Combined Statistical Area',
 'New England City and Town Area Division',
 'Urban Area',
 'Current Congressional District ***',
 'State Legislative District Upper',
 'State Legislative District Lower',
 'ZIP Code Tabulation Area (3-digit)',
 '5-digit ZIP Code Tabulation Area',
 'Subminor Civil Division (FIPS)',
 'State-School District (Elementary)',
 'State-School District (Secondary)',
 'State-School District (Unified)',
 'Urban/Rural',
 'Principal City Indicator',
 'Public Use Microdata Area – File',
 'Geographic Identifier',
 #'Area Name',
 'Tribal Tract',
 'Tribal Block Group',
 'Place (State FIPS + Place FIPS)', 'Total Population:',], axis=1)

# check 
print(acs_trim.columns)

Index(['FIPS', 'Census Tract', 'Area Name', 'Workers 16 Years and Over:',
       'Workers 16 Years and Over: Car, Truck, or Van',
       'Workers 16 Years and Over: Drove Alone',
       'Workers 16 Years and Over: Carpooled',
       'Workers 16 Years and Over: Public Transportation (Includes Taxicab)',
       'Workers 16 Years and Over: Motorcycle',
       'Workers 16 Years and Over: Bicycle',
       ...
       'Occupied Housing Units: 2 Vehicles Available',
       'Occupied Housing Units: 3 Vehicles Available',
       'Occupied Housing Units: 4 Vehicles Available',
       'Occupied Housing Units: 5 or More Vehicles Available',
       '% Occupied Housing Units: No Vehicle Available',
       '% Occupied Housing Units: 1 Vehicle Available',
       '% Occupied Housing Units: 2 Vehicles Available',
       '% Occupied Housing Units: 3 Vehicles Available',
       '% Occupied Housing Units: 4 Vehicles Available',
       '% Occupied Housing Units: 5 or More Vehicles Available'],
      dtype='o

## Next steps for ACS in this section

1. Remove columns 1-47 (show why they were removed through either being completely empty or unnecesary geographic components 
2. Remove census tracts where the total population is 0
3. Connect geojson component to graph (from week 3 census)


In [8]:
# remove columns
#acs_trim = acs.drop(acs.columns[1:46], axis=1)

In [9]:
# check new shape 
acs_trim.shape

(244, 109)

There are now 109 columns. All of these columns may not be necessary, but we are leaving them for now because we may need them later. 

## Trimming Rows
First, we will be verifying the data types of the variables left. While importing the data with column names included made data cleaning easier, the row of data (previously row 0) with the real variable changed the data types to object. This prevents us from conducting numeric extractions or sorting as we usually would. To combat this, we are changing the datatypes to numeric as applicable using a function.   

In [ ]:
# get info about data types 
acs_trim.info(verbose=True, show_counts=True)

In [ ]:
# make copy before converting 
trim = acs_trim.copy()

# convert data types to INT and drop columns with all NA values 
for col in trim.columns:
    if trim[col].dtype == 'object':
        trim[col] = pd.to_numeric(trim[col], errors='coerce').dropna(how='all').astype(int)

# verify
trim.dtypes

Now that the data types have converted to their normal types, we can continue data cleaning. 

## Add Geographic element for mapping 

Now, we are going to merge our cleaned dataframe to the geojson version of SF's census tracts, allowing us to map this data. We will merge on the `FIPS` column. 

In [ ]:
# import 
tracts = gpd.read_file("data/sf_2020_censusTract_data.geojson")

# check
tracts.head(1)

In [ ]:
# check shape 
tracts.shape

There are 244 rows (matching our data) and 15 columns. We need to remove all columns expect for `geo_id` (aka FIPS) and `geometry`, which allows us to map. These are the last two columns. 

In [ ]:
# import 
tracts = gpd.read_file("data/sf_2020_censusTract_data.geojson")

# check 
tracts.head(2)

In [ ]:
# remove extra columns
tracts_trim = tracts.drop(tracts.columns[0:13], axis=1)

# check 
tracts_trim

In [ ]:
# rename 'geoid' to FIPS to merge
tracts_trim = tracts_trim.rename(columns={'geoid': 'FIPS'})

tracts_trim.head(1)

In [ ]:
# change FIPS column to integer to merge
tracts_trim['FIPS'] = tracts_trim['FIPS'].astype(int)

# check 
print(tracts_trim.dtypes)

In [ ]:
# merge datasets
sf = tracts_trim.merge(trim,on="FIPS")

# check 
sf.head(1)

Successful merge! 

Now, we will be filtering out census tracts without any residents because they will not have any information useful to this project. To do this, we will be using the `Total Population` variable and dropping census tracts without any people living there. 

In [ ]:
# check empty census tracts 
sf[sf['Total Population'] == 0]

Only 3 rows have populations of 0. Now, we will drop them. 

In [ ]:
# create a new dataframe for census tracts with zero pop
df = sf[sf["Total Population"] != 0]

# check 
df[df["Total Population"] == 0]

Our data cleaning is mostly complete. There are still too many columns, as about half tell us the percentages in addition to the raw cases. 

For now, we are doing to leave `df` as the main dataframe and make subsets of this as needed. 

In [ ]:
# save cleaned data for later usage
df.to_file("clean_acs.geojson", driver="GeoJSON")

Now, we are going to look at the survey item `Means of Transportation to Work for Workers 16 Years and Over`. This will tell us the dominate modes of transportation used in SF.

Options: 
 'Workers 16 Years and Over:',
 'Workers 16 Years and Over: Car, Truck, or Van',
 'Workers 16 Years and Over: Drove Alone',
 'Workers 16 Years and Over: Carpooled',
 'Workers 16 Years and Over: Public Transportation (Includes Taxicab)',
 'Workers 16 Years and Over: Motorcycle',
 'Workers 16 Years and Over: Bicycle',
 'Workers 16 Years and Over: Walked',
 'Workers 16 Years and Over: Other Means',
 'Workers 16 Years and Over: Worked At Home',
 '% Workers 16 Years and Over: Car, Truck, or Van',
 '% Workers 16 Years and Over: Drove Alone',
 '% Workers 16 Years and Over: Carpooled',
 '% Workers 16 Years and Over: Public Transportation (Includes Taxicab)',
 '% Workers 16 Years and Over: Motorcycle',
 '% Workers 16 Years and Over: Bicycle',
 '% Workers 16 Years and Over: Walked',
 '% Workers 16 Years and Over: Other Means',
 '% Workers 16 Years and Over: Worked At Home,
 'Total Population'

In [ ]:
transport = df.copy

In [ ]:
# select tranportation columns and rename 
transport = df[[
 'FIPS',
 'Total Population',
 'Workers 16 Years and Over: Car, Truck, or Van',
 'Workers 16 Years and Over: Drove Alone',
 'Workers 16 Years and Over: Carpooled',
 'Workers 16 Years and Over: Public Transportation (Includes Taxicab)',
 'Workers 16 Years and Over: Motorcycle',
 'Workers 16 Years and Over: Bicycle',
 'Workers 16 Years and Over: Walked',
 'Workers 16 Years and Over: Other Means',
 'Workers 16 Years and Over: Worked At Home',
 'geometry']].set_axis([
     'FIPS',
     'Total Population',
     'Car, Truck, or Van',
     'Drove Alone',
     'Carpooled',
     'Public Transportation',
     'Motorcycle',
     'Bicycle',
     'Walked',
     'Other Means',
     'Worked At Home',
     'geometry'],
                       axis=1)


# Display the subset
transport.head(1)

Using the transportation subset, we are going to make a variable that groups all non-private vehicle transportation options, excluding the options to 'work from home' and 'motorcycle'.

This will tell us how many people in San Francisco are relying on alternative forms of transportation to get to work. While this variable does not tell us directly how many children use ublic transportation to get to school, it gives us insight on if there are dominant forms of transportation in specfic areas in SF. 

In [ ]:
# make a column for non-car transportation methods
#transport["% Non-Private"] = transport["% Public Transportation"] + transport["% Bicycle"] + transport["% Walked"] + transport["% Other Means"] + transport["% Carpooled"] 
transport["Non-Private Transport"] = transport["Public Transportation"] + transport["Bicycle"] + transport["Walked"] + transport["Other Means"] + transport["Carpooled"] 

#verify
transport.head(3)

In [ ]:
# calculate total number using non-private transportation methods
total_public = transport["Non-Private Transport"].sum()

# make object for total population
totalpop = transport["Total Population"].sum()

# make percentage for total percentage using non-private transportation
total_transit_percentage = round((total_public / totalpop) * 100)

# print as sentence
print("The total percentage of people using public transit in the city is %", total_transit_percentage)

About 25% of the San Francisco population uses non-private transportation methods to get to work, nearly **one fourth** of the total population. While this information does not directly tell us how many children use public transportation to get to SF public high schools, it gives us insight into the importance of public transportation methods to the general SF population.  

##  Public Transportation Usage

* NOTE: subtract work from home from total population because they arent using any transportation
* use the equity neighbhoods data
* make a function that shows the percentage of public transportation users in each neighborhoods
* BONUS: make a function that shows the percentage of children in these neighboroods (see group assingment #3 notebook)

In [ ]:
# import SF neighborhood data 
hoods = gpd.read_file("data/SF_Find_Neighborhoods_20250314.geojson")

# verify
hoods.head(5)

In [ ]:
# maps neighborhoods 
hoods.plot(figsize=(12,12))

In [ ]:
# spatial join transportation to neighborhoods 
transport_by_hood = gpd.sjoin(hoods, transport) #Spatial join points to polygons

# verify 
transport_by_hood.head(5)

In [ ]:
# choropleth map of arrests by neighborhood
fig,ax = plt.subplots(figsize=(12,12)) # empty plot

transport_by_hood.plot(ax=ax,column='Non-Private Transport',legend=True,cmap='Greens')